In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lightcurvelynx.consts import GAUSS_EFF_AREA2FWHM_SQ
from lightcurvelynx.obstable.ztf_obstable import ZTFObsTable, _ztfcam_ccd_gain, _ztfcam_readout_noise
from lightcurvelynx.astro_utils.mag_flux import mag2flux
from lightcurvelynx.astro_utils.noise_model import poisson_bandflux_std
import sqlite3

In [ ]:
con = sqlite3.connect("data/ztf_metadata_latest.db")
sql_query = "SELECT * FROM exposures"
metadata_table = pd.read_sql_query(sql_query, con)
metadata_table = metadata_table.replace("", np.nan)
metadata_table = metadata_table.dropna(subset=["fwhm"])
metadata_table.columns

In [ ]:
metadata_table

In [ ]:
obs_log = pd.read_parquet('ztfsniadr2/tables/observing_logs.parquet')
# assign median of zp across ccd for that expid
obs_log = obs_log.assign(
    zp = obs_log.groupby(['expid'])['zp'].transform('median')
)
obs_log = obs_log.assign(
    maglimit = obs_log.groupby(['expid'])['maglimit'].transform('median')
)
obs_log = obs_log.drop_duplicates("expid")

In [ ]:
obs_log

In [ ]:
obs_log.mjd.min(),obs_log.mjd.max()

In [ ]:
obs_log["filter"] = obs_log.apply(lambda row: row["band"][-1],axis=1)
obs_log = pd.merge(obs_log, metadata_table[["expid","filter","exptime","fwhm","obsdate","scibckgnd","ra","dec","maglim"]],on=["filter","expid"])
gain = _ztfcam_ccd_gain
obs_log["zp_nJy"] = mag2flux(obs_log["zp"].values + 2.5*np.log10(gain))
obs_log = obs_log.rename(columns={"zp":"zp_abmag"})

In [ ]:
obs_log

In [ ]:
# remove wrong ra/dec values 
print("before:",len(obs_log))
idx = obs_log.ra<0
display(obs_log.loc[idx])
obs_log = obs_log.loc[~idx]
print("after",len(obs_log))

In [ ]:
obs_log.infobits.unique()

In [ ]:
plt.hist(np.log10(obs_log.infobits+1))

In [ ]:
# remove bad quality epochs
# page 4 of https://irsa.ipac.caltech.edu/data/ZTF/docs/ztf_extended_cautionary_notes.
# " If INFOBITS for an image has value < 33554432 (i.e., does not contain bit 25), the image and catalog data are probably usable. "

# but ztf sn data has a flag to remove all that have infobits > 0. let's try that.

print("before:", len(obs_log))
# obs_log = obs_log.loc[obs_log.infobits < 33554432]
obs_log = obs_log.loc[obs_log.infobits == 0]
print("after:", len(obs_log))

In [ ]:
# let's try to derive skynoise using maglim and zp
# snr = flux/fluxerr
# fluxerr = sqrt(flux + sky_adu*npix*gain + readnoise**2*nexposure*npix + darkcurrent*npix*exptime*nexposure)
# 5 = flux/fluxerr
# 25 = flux**2/(flux + sky_adu*npix*gain + readnoise**2*nexposure*npix + darkcurrent*npix*exptime*nexposure)
# flux**2 - 25*flux -25*( sky_adu*npix*gain
#                         + readnoise**2*nexposure*npix
#                         + darkcurrent*npix*exptime*nexposure)
#                     = 0
# flux_e = 10^(-0.4*(maglim-zp))*gain
# sky_adu*npix*gain =  (flux_e**2 / 25 - flux_e - (readnoise**2*nexposure*npix
#                              + darkcurrent*npix*exptime*nexposure))
def compute_sky(row):
    gain = _ztfcam_ccd_gain
    nea = GAUSS_EFF_AREA2FWHM_SQ * (row["fwhm"]) ** 2
    flux = np.power(10., -0.4*(row['maglimit'] - row['zp_abmag'])) * gain
    sky = (flux**2 / 25 - flux - _ztfcam_readout_noise**2 * nea) / nea / gain
    return sky                          

In [ ]:
obs_log["sky_adu"] = obs_log.apply(compute_sky,axis=1)

In [ ]:
obs_log

In [ ]:
obs_log.to_parquet("data/ztf_observing_log_combined_w_metadata.parquet")

In [ ]:
obs_log.columns

In [ ]:
def calculate_median_cadence(mjd):
    mjd = np.sort(mjd)
    return np.median(np.diff(mjd))

In [ ]:
median_cadence_per_field = obs_log.groupby(['fieldid','band'])["mjd"].apply(calculate_median_cadence)
median_cadence_per_field.median()

In [ ]:
median_cadence_per_field.hist(bins=np.linspace(0,5))

In [ ]:
nea = GAUSS_EFF_AREA2FWHM_SQ * (obs_log["fwhm"]) ** 2

plt.plot(np.sqrt(obs_log["sky_adu"]*6.2*nea),np.sqrt(obs_log["sky_adu"]*6.2*nea)/(obs_log["skynoise"]*6.2),'o',alpha=0.5)
plt.axhline(y=1,c='k')
plt.show()
bins = np.linspace(0,1e3,20)
plt.hist(np.sqrt(obs_log["sky_adu"]*6.2*nea),bins=bins,alpha=0.5,label='sky noise from sky count',histtype='step')
plt.hist(obs_log["skynoise"]*6.2,bins=bins,alpha=0.5,label='sky noise from ztf sn',histtype='step') #skynoise = 1/5 * 10^(-0.4(maglim - magzp)) in ADU
plt.legend()

In [ ]:
# from astropy import units as u
# from astropy.coordinates import SkyCoord
# c = SkyCoord(ra=obs_log.fieldra.values*u.radian, dec=obs_log.fielddec.values*u.radian, frame='icrs')
# obs_log['fieldra_deg'] = c.ra.degree
# obs_log['fileddec_deg'] = c.dec.degree
# print(np.sum(np.isclose(c.ra.degree,obs_log.ra,atol=0.005)))
# print(np.sum(np.isclose(c.dec.degree,obs_log.dec,atol=0.005)))
# print(len(obs_log))

In [ ]:
# idx = np.isclose(c.ra.degree,obs_log.ra,atol=0.005)
# obs_log.loc[~idx]